# PCA Trajectory Plot

In [ ]:
# ============================
# PCA master (LLM: report+parameters; ML: best/worst per seed)
# ============================

import os, re, random, glob
from typing import Optional, Sequence, Union, Dict, Any, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib import cm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns  # for exact palette matching

# ignore warnings
import warnings
warnings.filterwarnings("ignore")

# -----------------------
# Dataset-specific observed columns
# -----------------------
# Normalize dataset_name and map to the correct "Observed ..." column
_OBS_COL_BY_DATASET = {
    # steels
    "matbench_steels": "Observed Yield Strength",
    "matbench_steels (composition)": "Observed Yield Strength",
    # membrane
    "membrane": "Observed Elastic Modulus",
    "membrane_dataset": "Observed Elastic Modulus",
    # p3ht
    "p3ht": "Observed Conductivity",
    "p3ht/cnt": "Observed Conductivity",
    "p3ht_dataset": "Observed Conductivity",
    # perovskite
    "perovskite": "Observed Instability Index",
    "perovskite_dataset": "Observed Instability Index",
}

def _norm_name(s: Optional[str]) -> Optional[str]:
    if s is None:
        return None
    # lower, trim, collapse spaces, remove some punctuation that often varies
    t = s.strip().lower()
    t = t.replace("_", " ").replace("-", " ").replace("/", "/")
    t = " ".join(t.split())
    return t

def observed_col_for_dataset(dataset_name: Optional[str]) -> Optional[str]:
    if dataset_name is None:
        return None
    key = _norm_name(dataset_name)
    if key in _OBS_COL_BY_DATASET:
        return _OBS_COL_BY_DATASET[key]
    # also try a few aliases (remove slashes)
    key2 = key.replace("/", " ")
    return _OBS_COL_BY_DATASET.get(key2, None)

# -----------------------
# Helpers: filename parsing + model-specific column mapping
# -----------------------
_MODEL_PATTERNS = [r"(?i)^(LLM)", r"(?i)^(GPR)", r"(?i)^(XGB)", r"(?i)^(RFR)", r"(?i)^(BNN)"]

def parse_trajectory_meta(path: str) -> Dict[str, Optional[str]]:
    """
    Extract model, seed, run (if any), alpha (if any), and flags (report/parameters/trajectory) from filename.
    """
    base = os.path.basename(path)

    # Model
    model = None
    for pat in _MODEL_PATTERNS:
        m = re.search(pat, base)
        if m:
            model = m.group(1).upper()
            break
    if model is None:
        model = base[:3].upper()

    # Seed / Run / Alpha
    m_seed  = re.search(r"(?i)seed(\d+)", base)
    m_run   = re.search(r"(?i)run(\d+)",  base)
    m_alpha = re.search(r"(?i)alpha([0-9]*\.?[0-9]+)", base)

    # Trajectory/report/parameters flags
    is_params = bool(re.search(r"(?i)parameters", base))
    is_report = bool(re.search(r"(?i)report", base))
    is_traj   = bool(re.search(r"(?i)trajectory", base))

    return {
        "model": model,
        "seed":  m_seed.group(1)  if m_seed  else None,
        "run":   m_run.group(1)   if m_run   else None,
        "alpha": m_alpha.group(1) if m_alpha else None,
        "is_params": is_params,
        "is_report": is_report,
        "is_trajectory": is_traj,
        "base": base,
    }

def cols_for_model(model: Optional[str]) -> tuple[str, str]:
    """
    Return (experiment_index_col, obs_y_in_traj_col) based on model family.
    LLM -> ('Experiment Index', 'Observed Yield Strength')
    Others -> ('Index', 'Observed Target Value')
    (Note: observed column may be overridden per dataset below.)
    """
    if (model or "").upper() == "LLM":
        return "Experiment Index", "Observed Yield Strength"
    else:
        return "Index", "Observed Target Value"

# -----------------------
# Aggregations: mean±std for any metric (we'll use CUM_L1)
# -----------------------
def aggregate_metric(dfs: List[pd.DataFrame], col: str = "CUM_L1") -> Tuple[np.ndarray, np.ndarray]:
    """
    Align by step index and return (mean, std) of the requested column.
    """
    if not dfs:
        return np.array([]), np.array([])
    max_len = max(len(df) for df in dfs)
    mat = np.full((len(dfs), max_len), np.nan)
    for r, df in enumerate(dfs):
        vals = df.sort_values("step")[col].to_numpy()
        mat[r, :len(vals)] = vals
    return np.nanmean(mat, axis=0), np.nanstd(mat, axis=0)

# -----------------------
# (kept, but not used for Random Walks now) Synthetic random walk generator
# -----------------------
def make_random_walk_traj(
    data: Union[str, pd.DataFrame],
    y_col: str,
    seed: int,
    index_is_one_based: bool = False,
) -> pd.DataFrame:
    data_df = pd.read_csv(data) if isinstance(data, str) else data.copy()
    valid_idx = data_df[y_col].dropna().index.to_list()
    if not valid_idx:
        raise ValueError("No valid rows with target present for random walk.")
    rng = random.Random(seed)
    np.random.seed(seed)
    y_vals = data_df.loc[valid_idx, y_col].to_numpy()
    best_pos = valid_idx[int(np.nanargmax(y_vals))]
    order = valid_idx[:]
    rng.shuffle(order)
    path = []
    for ridx in order:
        path.append(ridx)
        if ridx == best_pos:
            break
    rows = []
    for it, ridx in enumerate(path):
        label = (ridx + 1) if index_is_one_based else ridx
        rows.append({
            "Iteration": it,
            "Index": label,
            "Observed Target Value": float(data_df.loc[ridx, y_col]),
        })
    return pd.DataFrame(rows)

# -----------------------
# NEW helpers for Random Walk files
# -----------------------
def _find_random_walk_files(folder: str) -> List[str]:
    """
    Look for existing Random Walk CSVs in a dataset folder using forgiving patterns.
    Includes:
      - <folder>/random_walk/**/*.csv
      - <folder>/random_walk*.csv
      - <folder>/**/*random*walk*.csv
    """
    patterns = [
        os.path.join(folder, "random_walk", "**", "*.csv"),
        os.path.join(folder, "random_walk*.csv"),
        os.path.join(folder, "**", "*random*walk*.csv"),
    ]
    seen, files = set(), []
    for pat in patterns:
        for p in glob.glob(pat, recursive=True):
            if os.path.isfile(p) and p not in seen:
                seen.add(p)
                files.append(os.path.abspath(p))
    return files

def _extract_seed_from_filename(path: str) -> Optional[str]:
    """
    Try to extract a seed from a filename like '...seed38...' or '...seed_39...'.
    Returns the numeric seed as a string, or None if not found.
    """
    base = os.path.basename(path)
    m = re.search(r"(?i)seed[_-]?(\d+)", base)
    if m:
        return m.group(1)
    # fallback: look for 's38' or '-38' when preceded by 'seed' text is missing — optional
    m2 = re.search(r"(?<!\d)(\d{1,4})(?!\d)", base)
    return m2.group(1) if m2 else None

# -----------------------
# Core: per-file computation + (optional) plotting (PCA)
# -----------------------
def pca_with_trajectory(
    data: Union[str, pd.DataFrame],
    traj: Union[str, pd.DataFrame],
    x_cols: Sequence[str],
    y_col: Union[str, Sequence[str]],   # tolerate 1-tuple/list
    goal: str = "max",
    experiment_index_col: str = "Experiment Index",
    obs_y_in_traj_col: str = "Observed Yield Strength",
    index_is_one_based: bool = False,
    mismatch_abs_tol: float = 1e-6,
    arrows: bool = True,
    annotate_every: int = 0,
    alpha_bg: float = 0.5,
    point_size_bg: int = 30,
    point_size_traj: int = 60,
    figsize: tuple = (12, 8),
    dataset_name: Optional[str] = None,
    n_samples: Optional[int] = None,
    title: Optional[str] = None,
    alpha_min: float = 0.15,
    alpha_max: float = 0.85,
    font_scale: float = 1.4,
    random_state: int = 0,
    meta: Optional[Dict[str, Optional[str]]] = None,
    show_plot: bool = True,
) -> Dict[str, Any]:
    """
    Compute PCA + trajectory + step_df (including cumulative distance).
    """

    # --- sanitize y_col / x_cols ---
    if isinstance(y_col, (list, tuple)):
        if len(y_col) != 1:
            raise ValueError(f"y_col must be a single column, got: {y_col}")
        y_col = y_col[0]
    if isinstance(x_cols, (str, bytes)):
        x_cols = [x_cols]
    elif isinstance(x_cols, tuple):
        x_cols = list(x_cols)

    # --- Load data/traj ---
    data_df = pd.read_csv(data) if isinstance(data, str) else data.copy()
    traj_df = pd.read_csv(traj) if isinstance(traj, str) else traj.copy()

    base = os.path.basename(traj) if isinstance(traj, str) else "trajectory"
    model_name = base[:3].upper()

    # --- If NON-LLM traj, synthesize Iteration 0 (same behavior as before) ---
    is_path = isinstance(traj, str)
    is_non_llm = (is_path and ("llm" not in str(traj).lower()))
    has_iter0  = ("Iteration" in traj_df.columns) and (traj_df["Iteration"].astype(str) == "0").any()
    m_seed_base = re.search(r"(?i)seed(\d+)", base) if isinstance(traj, str) else None
    if is_non_llm and not has_iter0:
        seed_val = int(m_seed_base.group(1)) if m_seed_base else 42
        rng = random.Random(seed_val)
        initial_pos = rng.choice(list(range(len(data_df))))
        initial_label = initial_pos + 1 if index_is_one_based else initial_pos
        y0 = data_df.iloc[initial_pos][y_col]
        init_row = {"Iteration": 0, experiment_index_col: initial_label, obs_y_in_traj_col: y0}
        for col in [experiment_index_col, obs_y_in_traj_col, "Iteration"]:
            if col not in traj_df.columns:
                traj_df[col] = np.nan
        if "Iteration" in traj_df.columns and traj_df["Iteration"].notna().any():
            traj_df["Iteration"] = pd.to_numeric(traj_df["Iteration"], errors="coerce").add(1)
        traj_df = pd.concat([pd.DataFrame([init_row]), traj_df], ignore_index=True)

    # --- Robust observed column selection per dataset ---
    desired_obs_col = obs_y_in_traj_col
    ds_obs = observed_col_for_dataset(dataset_name)
    if ds_obs is not None:
        desired_obs_col = ds_obs

    if desired_obs_col not in traj_df.columns:
        candidates = [c for c in traj_df.columns if isinstance(c, str) and c.strip().lower().startswith("observed")]
        picked = None
        if ds_obs and candidates:
            key_token = ds_obs.replace("Observed", "").strip().lower()
            for c in candidates:
                if key_token and key_token in c.lower():
                    picked = c
                    break
        if picked is None and candidates:
            picked = sorted(candidates, key=len)[-1]
        if picked:
            desired_obs_col = picked
        else:
            print(f"[warn] Could not find an observed column in traj_df for dataset '{dataset_name}'. "
                  f"Tried '{obs_y_in_traj_col}' and '{ds_obs}'. Using NaNs for colors.")

    # --- PCA embedding ---
    X = data_df[x_cols].select_dtypes(include=[np.number])
    valid_rows = X.dropna().index.intersection(data_df[y_col].dropna().index)
    scaler = StandardScaler().fit(X.loc[valid_rows])
    X_scaled = scaler.transform(X.loc[valid_rows])

    pca = PCA(n_components=2, random_state=random_state)
    emb = pca.fit_transform(X_scaled)  # (n_valid, 2)

    data_df["PCA1"] = np.nan
    data_df["PCA2"] = np.nan
    data_df.loc[valid_rows, ["PCA1", "PCA2"]] = emb
    y_vals_all = data_df.loc[valid_rows, y_col].to_numpy()

    vmin, vmax = np.nanmin(y_vals_all), np.nanmax(y_vals_all)
    cmap = cm.get_cmap("viridis")
    norm = Normalize(vmin=vmin, vmax=vmax)

    # --- Build trajectory indices robustly ---
    idx_series  = pd.to_numeric(traj_df.get(experiment_index_col), errors="coerce")
    iter_series = pd.to_numeric(traj_df.get("Iteration", pd.Series(range(len(traj_df)))), errors="coerce")

    valid_mask = idx_series.notna()
    idx_vals   = idx_series.where(valid_mask).dropna().astype(int).to_numpy()
    iter_vals  = iter_series.where(valid_mask).dropna().to_numpy()

    if index_is_one_based:
        idx_vals = idx_vals - 1

    # Keep only in-bounds rows that also have PCA coords
    valid_pos, iter_list = [], []
    for i_lbl, itv in zip(idx_vals, iter_vals):
        if 0 <= i_lbl < len(data_df):
            if pd.notna(data_df.iloc[i_lbl]["PCA1"]) and pd.notna(data_df.iloc[i_lbl]["PCA2"]):
                valid_pos.append(i_lbl)
                iter_list.append(itv)

    traj_points = data_df.iloc[valid_pos].copy()

    # Observed values aligned to valid_pos order (used for segment color)
    obs_vals = []
    for i_lbl in valid_pos:
        search_val = i_lbl + 1 if index_is_one_based else i_lbl
        rows = traj_df.get(experiment_index_col) == search_val
        if (desired_obs_col in traj_df.columns) and rows.any():
            obs_vals.append(traj_df.loc[rows, desired_obs_col].iloc[0])
        else:
            obs_vals.append(np.nan)
    obs_vals = np.array(obs_vals)

    # Step metrics (+ cumulative) in PCA space
    x = traj_points["PCA1"].to_numpy()
    y = traj_points["PCA2"].to_numpy()
    y_true = data_df.iloc[traj_points.index][y_col].to_numpy()

    if len(x) >= 2:
        dx = np.diff(x); dy = np.diff(y)
        l1_steps = np.abs(dx) + np.abs(dy)
        l2_steps = np.sqrt(dx**2 + dy**2)
        step_df = pd.DataFrame({
            "step": np.arange(1, len(x)),
            "L1_PCA": l1_steps,
            "L2_PCA": l2_steps,
        })
        step_df["CUM_L1"] = step_df["L1_PCA"].cumsum()
        step_df["CUM_L2"] = step_df["L2_PCA"].cumsum()
    else:
        step_df = pd.DataFrame(columns=["step", "L1_PCA", "L2_PCA", "CUM_L1", "CUM_L2"])

    fig = ax = None
    if show_plot:
        # --- Base scatter ---
        fig, ax = plt.subplots(figsize=figsize, dpi=300, facecolor="white")
        ax.scatter(
            data_df.loc[valid_rows, "PCA1"],
            data_df.loc[valid_rows, "PCA2"],
            c=y_vals_all, cmap=cmap, norm=norm,
            alpha=alpha_bg, s=point_size_bg,
        )
        # no colorbar to match your current style

        ax.set_xlabel("PC 1", fontsize=14 * font_scale)
        ax.set_ylabel("PC 2", fontsize=14 * font_scale)

        # Title bits (include model/seed/run/alpha + report/parameters for LLM)
        name = dataset_name or "Dataset"
        n_str = f"(n={n_samples})" if n_samples is not None else ""
        extra_bits = []
        if meta:
            if meta.get("model"): extra_bits.append(meta["model"])
            if meta.get("seed"):  extra_bits.append(f"Seed {meta['seed']}")
            if meta.get("run"):   extra_bits.append(f"Run {meta['run']}")
            if meta.get("alpha"): extra_bits.append(f"alpha={meta['alpha']}")
            if meta.get("model") == "LLM":
                if meta.get("is_report"):  extra_bits.append("(report)")
                if meta.get("is_params"):  extra_bits.append("(parameters)")
        extra = " [" + ", ".join(extra_bits) + "]" if extra_bits else ""
        title_str = title or f"{name}{extra} {n_str}".strip()
        ax.set_title(title_str, fontsize=16 * font_scale, pad=12)
        ax.grid(alpha=0.3)
        ax.tick_params(axis="both", which="major", labelsize=12 * font_scale)

        # Segments/arrows/annotations
        if len(x) >= 2:
            segments = np.stack(
                [np.column_stack([x[:-1], y[:-1]]),
                 np.column_stack([x[1:],  y[1:]])],
                axis=1,
            )
            nseg = segments.shape[0]
            alphas = np.linspace(alpha_min, alpha_max, nseg)

            # Robust color handling (avoid gray if obs_vals is NaN/inf)
            colors = []
            for i in range(nseg):
                val = obs_vals[i+1] if i+1 < len(obs_vals) else np.nan
                if np.isnan(val) or np.isinf(val):
                    rgba = (0.3, 0.3, 0.3, alphas[i])   # fallback gray
                else:
                    rgba = (*cmap(norm(val))[:3], alphas[i])
                colors.append(rgba)

            lc = LineCollection(segments, colors=colors, linewidths=1.2, zorder=2)
            ax.add_collection(lc)

            if arrows:
                for i in range(nseg):
                    (x0, y0), (x1, y1) = segments[i]
                    ax.arrow(
                        x0, y0, x1 - x0, y1 - y0,
                        shape="full", lw=0, length_includes_head=True,
                        head_width=0.15, head_length=0.15,
                        color=colors[i], zorder=3,
                    )

            if annotate_every and annotate_every > 0:
                for k in range(0, len(x), annotate_every):
                    xv, yv = x[k], y[k]
                    ax.annotate(
                        str(k), (xv, yv),
                        xytext=(6, 6), textcoords="offset points",
                        fontsize=10 * font_scale, color="black",
                        alpha=alphas[min(max(k-1,0), len(alphas)-1)] if k>0 else 0.2,
                        bbox=dict(boxstyle="round,pad=0.25", facecolor="lightgray", alpha=0.3, edgecolor="none"),
                    )

        # Points + start + best (legend "Best (iter #)")
        if len(x) > 0:
            ax.scatter(x, y, s=point_size_traj, alpha=0.95, c=y_true, cmap=cmap, norm=norm, zorder=4)
            ax.scatter(x[0], y[0], s=180, marker="s", facecolors="none",
                       edgecolor="black", lw=1.5, label="Start (Iter 0)", zorder=5)

            if len(y_true) > 0 and np.isfinite(y_true).any():
                best_local_idx = int(np.nanargmax(y_true) if goal == "max" else np.nanargmin(y_true))
                if len(iter_list) == len(traj_points) and pd.notna(iter_list[best_local_idx]):
                    best_iter_label = int(iter_list[best_local_idx])
                else:
                    best_iter_label = best_local_idx
                ax.scatter(x[best_local_idx], y[best_local_idx], s=260, marker="*",
                           color="gold", edgecolor="black", lw=1.5,
                           label=f"Best (iter {best_iter_label})", zorder=6)

            ax.legend(frameon=True, loc="upper left", fontsize=12 * font_scale)
    else:
        if plt.get_fignums():
            plt.close('all')

    return {
        "fig": fig,
        "ax": ax,
        "pca": pca,
        "step_df": step_df,     # includes CUM_L1 / CUM_L2 in PCA space
        "meta": meta or {},
        "path": traj if isinstance(traj, str) else None,
        "num_iters": int(step_df["step"].max() + 1) if len(step_df) else 0,
    }

# -----------------------
# Master: scan, compute (quiet), choose, then render (PCA)
# -----------------------
def pca_master_all_types(
    folder: str,
    data_csv: Union[str, pd.DataFrame],
    x_cols: Sequence[str],
    y_col: Union[str, Sequence[str]],
    # behavior
    goal: str = "max",
    # styling
    annotate_every: int = 5,
    alpha_bg: float = 0.5,
    point_size_bg: int = 30,
    point_size_traj: int = 60,
    figsize: tuple = (12, 8),
    dataset_name: Optional[str] = "matbench_steels",
    n_samples: Optional[int] = 312,
    font_scale: float = 1.4,
    # generic seed for initial-iteration synthesis (non-LLM files)
    random_state: int = 0,
    # legacy arg kept for signature compat
    random_walk_seeds: Sequence[int] = (38, 39, 40, 41, 42),
) -> Dict[str, Any]:
    """
    PCA version of the master:
    - Include BOTH LLM report and parameters files.
    - Plot 4 LLM trajectories: best/worst within 'report' and within 'parameters'.
    - For EACH ML model and for EACH seed found, plot best+worst.
    - Add Random Walk trajectories (READ FROM FILES) and render them; titles include seed.
    """

    # sanitize once at entry
    if isinstance(y_col, (list, tuple)):
        if len(y_col) != 1:
            raise ValueError(f"y_col must be a single column, got: {y_col}")
        y_col = y_col[0]
    if isinstance(x_cols, (str, bytes)):
        x_cols = [x_cols]
    elif isinstance(x_cols, tuple):
        x_cols = list(x_cols)

    folder = os.path.abspath(folder)
    csv_paths = glob.glob(os.path.join(folder, "*.csv"))

    # Filter to trajectories (any model). Include ALL LLM types (report + parameters).
    selected_files: List[str] = []
    metas: Dict[str, Dict[str, Optional[str]]] = {}
    for p in csv_paths:
        meta = parse_trajectory_meta(p)
        if not meta["is_trajectory"]:
            continue
        selected_files.append(p)
        metas[p] = meta

    if not selected_files:
        print("[warn] No trajectory files found.]")

    # Pass 1: compute quietly (file-based)
    results_quiet: Dict[str, Dict[str, Any]] = {}
    steps_by_model: Dict[str, List[pd.DataFrame]] = {}
    iters_by_path: Dict[str, int] = {}

    for traj_path in sorted(selected_files):
        meta = metas[traj_path]
        exp_idx_col, obs_col = cols_for_model(meta.get("model"))

        # Override observed column for known datasets (LLM)
        ds_override = observed_col_for_dataset(dataset_name)
        if ds_override and meta.get("model") == "LLM":
            obs_col = ds_override

        out = pca_with_trajectory(
            data=data_csv,
            traj=traj_path,
            x_cols=x_cols,
            y_col=y_col,
            goal=goal,
            experiment_index_col=exp_idx_col,
            obs_y_in_traj_col=obs_col,
            annotate_every=annotate_every,
            alpha_bg=alpha_bg,
            point_size_bg=point_size_bg,
            point_size_traj=point_size_traj,
            figsize=figsize,
            dataset_name=dataset_name,
            n_samples=n_samples,
            font_scale=font_scale,
            random_state=random_state,
            meta=meta,
            show_plot=False,  # compute silently
        )
        results_quiet[traj_path] = out

        model = meta["model"]
        if model == "LLM":
            if meta.get("is_report"):
                model_key = "LLM (report)"
            elif meta.get("is_params"):
                model_key = "LLM (parameters)"
            else:
                model_key = "LLM"
        else:
            model_key = model

        if model_key and len(out["step_df"]) > 0:
            steps_by_model.setdefault(model_key, []).append(
                out["step_df"][["step", "L1_PCA", "CUM_L1"]].copy()
            )
            iters_by_path[traj_path] = out["num_iters"]
        else:
            iters_by_path[traj_path] = 0

    # Helper to render one (file-based)
    def render_one(path: str):
        meta = metas[path]
        exp_idx_col, obs_col = cols_for_model(meta.get("model"))
        ds_override = observed_col_for_dataset(dataset_name)
        if ds_override and meta.get("model") == "LLM":
            obs_col = ds_override
        return pca_with_trajectory(
            data=data_csv,
            traj=path,
            x_cols=x_cols,
            y_col=y_col,
            goal=goal,
            experiment_index_col=exp_idx_col,
            obs_y_in_traj_col=obs_col,
            annotate_every=annotate_every,
            alpha_bg=alpha_bg,
            point_size_bg=point_size_bg,
            point_size_traj=point_size_traj,
            figsize=figsize,
            dataset_name=dataset_name,
            n_samples=n_samples,
            font_scale=font_scale,
            random_state=random_state,
            meta=meta,
            show_plot=True,  # now render
        )

    rendered_results: List[Dict[str, Any]] = []

    # =========================
    # LLM: 4 plots
    # =========================
    llm_paths = [p for p in selected_files if metas[p]["model"] == "LLM"]
    llm_report_paths = [p for p in llm_paths if metas[p]["is_report"]]
    llm_param_paths  = [p for p in llm_paths if metas[p]["is_params"]]

    def render_best_worst(paths: List[str], label: str):
        if not paths:
            print(f"[warn] No LLM {label} files found.")
            return
        sorted_paths = sorted(paths, key=lambda p: iters_by_path.get(p, 0))
        best_path = sorted_paths[0]
        worst_path = sorted_paths[-1]
        print(f"[LLM-{label}] Best:  {os.path.basename(best_path)}  (iters={iters_by_path.get(best_path,0)})")
        print(f"[LLM-{label}] Worst: {os.path.basename(worst_path)} (iters={iters_by_path.get(worst_path,0)})")
        rendered_results.append(render_one(best_path))
        if worst_path != best_path:
            rendered_results.append(render_one(worst_path))

    render_best_worst(llm_report_paths, "report")
    render_best_worst(llm_param_paths,  "parameters")

    # =========================
    # ML models: for EACH seed present in that model, plot best & worst
    # =========================
    ml_models = ["GPR", "XGB", "RFR", "BNN"]
    for model in ml_models:
        model_paths = [p for p in selected_files if metas[p]["model"] == model]
        if not model_paths:
            print(f"[warn] No files for model {model}.")
            continue

        seeds = sorted({metas[p]["seed"] for p in model_paths if metas[p]["seed"] is not None})
        if not seeds:
            print(f"[warn] No seeds found for model {model}.")
            continue

        for seed in seeds:
            candidates = [p for p in model_paths if metas[p]["seed"] == seed]
            if not candidates:
                continue
            cand_sorted = sorted(candidates, key=lambda p: iters_by_path.get(p, 0))
            best_path  = cand_sorted[0]
            worst_path = cand_sorted[-1]
            print(f"[{model}] Seed {seed}: best={os.path.basename(best_path)} (iters={iters_by_path.get(best_path,0)}), "
                  f"worst={os.path.basename(worst_path)} (iters={iters_by_path.get(worst_path,0)})")
            rendered_results.append(render_one(best_path))
            if worst_path != best_path:
                rendered_results.append(render_one(worst_path))

    # =========================
    # Random-walk baselines (READ FROM FILES) — title shows seed
    # =========================
    rw_files = _find_random_walk_files(folder)
    if not rw_files:
        print(f"[warn] No random-walk files found under: {folder}")

    for rw_path in sorted(rw_files):
        seed_str = _extract_seed_from_filename(rw_path)
        meta_rw = {"model": "Random Walk", "seed": seed_str, "is_trajectory": True}
        out_rw = pca_with_trajectory(
            data=data_csv,
            traj=rw_path,                 # read the precomputed Random Walk CSV
            x_cols=x_cols,
            y_col=y_col,
            goal=goal,
            experiment_index_col="Index",             # RW uses ML-style columns
            obs_y_in_traj_col="Observed Target Value",
            annotate_every=annotate_every,
            alpha_bg=alpha_bg,
            point_size_bg=point_size_bg,
            point_size_traj=point_size_traj,
            figsize=figsize,
            dataset_name=dataset_name,
            n_samples=n_samples,
            font_scale=font_scale,
            random_state=random_state,
            meta=meta_rw,
            show_plot=True,
        )
        rendered_results.append(out_rw)

    # NOTE: distance plots have been removed per your request

    return {
        "rendered_results": rendered_results,
        "results_quiet": results_quiet,
        "steps_by_model": steps_by_model,   # left for compatibility (unused if you don't plot)
        "fig_all_models": None,
        "ax_all_models": None,
        "per_model_figs": {},
    }


In [ ]:
if __name__ == "__main__":
    folder = r"al_trajectory_data_all\matbench_steels (composition)"  # <- your folder
    data_csv = "matbench_steels.csv"

    x_cols = ['Fe','C','Mn','Si','Cr','Ni','Mo','V','Nb','Co','Al','Ti','N','W']
    y_col = "yield strength"  # target column in the dataset CSV

    out = pca_master_all_types(
        folder=folder,
        data_csv=data_csv,
        x_cols=x_cols,
        y_col=y_col,
        goal="max",
        annotate_every=5,         # keep your annotation cadence
        alpha_bg=0.5,
        point_size_bg=30,
        point_size_traj=60,
        figsize=(8, 4),
        dataset_name="matbench_steels",
        n_samples=312,
        font_scale=1,           # keep your font sizes
        random_state=0,
    )

In [ ]:
# -----------------------
# Example usage
# -----------------------
if __name__ == "__main__":
    folder = r"al_trajectory_data_all\P3HT_dataset"  # <- your folder
    data_csv = "P3HT_dataset.csv"
    x_cols=['P3HT content (%)', 'D1 content (%)', 'D2 content (%)',
       'D6 content (%)', 'D8 content (%)']
    y_col='Conductivity (measured) (S/cm)'

    out = pca_master_all_types(
        folder=folder,
        data_csv=data_csv,
        x_cols=x_cols,
        y_col=y_col,
        goal="max",
        annotate_every=5,         # keep your annotation cadence
        alpha_bg=0.5,
        point_size_bg=30,
        point_size_traj=60,
        figsize=(8, 4),
        dataset_name="P3HT/CNT",
        n_samples=323,
        font_scale=1,           # keep your font sizes
        random_state=0,
    )

In [ ]:
# -----------------------
# Example usage
# -----------------------
if __name__ == "__main__":
    folder = r"al_trajectory_data_all\Perovskite_dataset"  # <- your folder
    data_csv = "Perovskite_dataset.csv"
    x_cols=['CsPbI', 'FAPbI', 'MAPbI']
    y_col="Instability index"

    out = pca_master_all_types(
        folder=folder,
        data_csv=data_csv,
        x_cols=x_cols,
        y_col=y_col,
        goal="min",
        annotate_every=5,         # keep your annotation cadence
        alpha_bg=0.5,
        point_size_bg=30,
        point_size_traj=60,
        figsize=(8, 4),
        dataset_name="Perovskite",
        n_samples=139,
        font_scale=1,           # keep your font sizes
        random_state=0,
    )

In [ ]:
# -----------------------
# Example usage
# -----------------------
if __name__ == "__main__":
    folder = r"al_trajectory_data_all\Membrane_dataset"  # <- your folder
    data_csv = "Membrane_dataset.csv"
    x_cols=['Auto', 'Heating', 'Concentration', 'Heating Block Temperature', 'Mixed', 'Rest Time (days)',
       'Coupon-to-Bath Wait Time (min)', 'Relative Humidity',
       'Nitrogen (Side from drop)', 'Nitrogen (After blade)']
    y_col="Elastic Modulus_mean"

    out = pca_master_all_types(
        folder=folder,
        data_csv=data_csv,
        x_cols=x_cols,
        y_col=y_col,
        goal="max",
        annotate_every=5,         # keep your annotation cadence
        alpha_bg=0.5,
        point_size_bg=30,
        point_size_traj=60,
        figsize=(8, 4),
        dataset_name="Membrane",
        n_samples=73,
        font_scale=1,           # keep your font sizes
        random_state=0,
    )